In [1]:
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

In [2]:
# ---------------------------
# Couplings with JAX-safe control
# ---------------------------
def make_couplings_jax(Lx, Ly, tx, ty, J=1.0):
    tx = jnp.asarray(tx)
    ty = jnp.asarray(ty)

    Jx = J * jnp.ones((Ly, Lx))
    Jy = J * jnp.ones((Ly, Lx))

    # antiperiodic seam along last row/column
    Jx = Jx.at[:, -1].multiply(1 - 2*tx)
    Jy = Jy.at[-1, :].multiply(1 - 2*ty)

    return Jx, Jy

# ---------------------------
# Energy computation
# ---------------------------
def compute_energies(spin_series, Jx, Jy, L):
    def single_energy(bits_single):
        spins = bits_single.reshape(L, L)
        spins = 2*spins - 1  # convert {0,1} -> {-1,+1}

        sx = jnp.roll(spins, -1, axis=1)
        sy = jnp.roll(spins, -1, axis=0)

        return -jnp.sum(Jx * spins * sx) - jnp.sum(Jy * spins * sy)

    return jax.vmap(single_energy)(spin_series)

# ---------------------------
# Transpose lattice (modular S)
# ---------------------------
def S_transform_bits(bits, L):
    return bits.reshape(-1, L, L).transpose(0,2,1).reshape(-1, L*L)

# ---------------------------
# Compute sector probabilities
# ---------------------------
def sector_probabilities(bits, beta, L):
    sectors = jnp.array([[0,0],[0,1],[1,0],[1,1]])

    def one_sector(sec):
        tx, ty = sec
        Jx, Jy = make_couplings_jax(L,L,tx,ty)
        E = compute_energies(bits, Jx, Jy, L)
        logw = -beta * E
        logZ = jax.scipy.special.logsumexp(logw)
        return jnp.exp(logw - logZ)  # normalized probability

    return jax.vmap(one_sector)(sectors)

# ---------------------------
# S-overlap matrix (real)
# ---------------------------
def S_overlap_matrix(bits, beta, L):
    psi = jnp.sqrt(sector_probabilities(bits, beta, L))
    psiS = jnp.sqrt(sector_probabilities(S_transform_bits(bits,L), beta, L))
    return psi @ psiS.T

In [7]:
# ---------------------------
# Run for L=4
# ---------------------------
L = 4
b_c = jnp.log(1+jnp.sqrt(2))/2

numbers = jnp.arange(2**(L*L), dtype=jnp.uint16)
bits = ((numbers[:, None] >> jnp.arange(L*L)) & 1)

# energies and Boltzmann weights for GSO
sectors = [(0,0),(0,1),(1,0),(1,1)]
energies = []
spinStateWts = []

for sector in sectors:
    Jx, Jy = make_couplings_jax(L,L,sector[0],sector[1])
    E = compute_energies(bits,Jx,Jy,L)
    spinStateWts.append(jnp.exp(-b_c * E))
    energies.append(E)

energies = jnp.array(energies)
spinStateWts = jnp.array(spinStateWts)

# ---------------------------
# GSO projection to fermion sectors
# ---------------------------
zFproject = jnp.array([
    [1,  1,  1,  1],  # NS, NS
    [1,  1, -1,  1],  # NS, R
    [1, -1,  1,  1],  # R, NS
    [1, -1, -1, -1]   # R, R
], dtype=jnp.float64)

# implement s-duality on the fermionic sector
S_F = jnp.array([[1,0,0,0],
                 [0,0,1,0],
                 [0,1,0,0],
                 [0,0,0,1]], dtype=jnp.float64)

In [8]:
Z_spin = jnp.sum(spinStateWts, axis=1)
Z_fermion = zFproject @ Z_spin
print(f"CFT Sector Partition Functions NSNS : {Z_fermion[0]}, NS-R: {Z_fermion[1]}, R-NS : {Z_fermion[2]}, RR: {Z_fermion[3]}")
s_dual_Z_fermion = S_F @ Z_fermion
print("Difference under S-duality:", s_dual_Z_fermion - Z_fermion)

CFT Sector Partition Functions NSNS : 11018239.999999989, NS-R: 6823935.999999993, R-NS : 6823935.999999993, RR: -1.862645149230957e-09
Difference under S-duality: [0. 0. 0. 0.]


In [57]:
# same sequence of transformations applied to spin_microstates
# only now decouple the RR mode from before
psi_fermion = (zFproject @ spinStateWts).astype(complex)
psi_fermion = psi_fermion[:-1,:]
# take square roots of fermions
wave_fns = jnp.sqrt(psi_fermion)

In [52]:
#overlap of un-normalized wave functions
overlaps = wave_fns@(jnp.conjugate(wave_fns.T))
# diagonal elements of over-laps are the norms of wave functions
norms = jnp.sqrt(jnp.diag(overlaps))
# normalized wave_functions, divide each component of wave function by sqrt norm
wave_fns = wave_fns/norms[:,None]
#overlap of normalized wave functions
overlaps = wave_fns@(jnp.conjugate(wave_fns.T))
jnp.round(overlaps,3)

Array([[1.   +0.j   , 0.861-0.113j, 0.861-0.113j],
       [0.861+0.113j, 1.   +0.j   , 0.688-0.j   ],
       [0.861+0.113j, 0.688+0.j   , 1.   +0.j   ]], dtype=complex128)

In [53]:
#diagonalizing the overlap matrix to extract the orthonormal basis
eigen, basis_change = jnp.linalg.eigh(overlaps)
orthogonal_basis = jnp.conjugate(basis_change.T) @ wave_fns
eigen

Array([0.06853845, 0.31167683, 2.61978472], dtype=float64)

In [54]:
jnp.round(jnp.conjugate(basis_change.T),2)

Array([[-0.8 -0.j  ,  0.42-0.06j,  0.42-0.06j],
       [ 0.  -0.j  ,  0.71-0.04j, -0.71+0.04j],
       [-0.6 -0.j  , -0.56+0.07j, -0.56+0.07j]], dtype=complex128)

In [55]:
jnp.round(orthogonal_basis@(jnp.conjugate(orthogonal_basis.T)),2)

Array([[ 0.07+0.j,  0.  +0.j, -0.  +0.j],
       [ 0.  -0.j,  0.31+0.j, -0.  -0.j],
       [-0.  -0.j, -0.  +0.j,  2.62+0.j]], dtype=complex128)

In [66]:
# same sequence of transformations applied to spin_microstates
# only now decouple the RR mode from before
psi_fermion = (zFproject @ spinStateWts).astype(complex)
psi_fermion = psi_fermion[:-1,:]
# take square roots of fermions
wave_fns = jnp.sqrt(psi_fermion)

#overlap of un-normalized wave functions
overlaps = wave_fns@(jnp.conjugate(wave_fns.T))
print(jnp.round(overlaps,2))
#diagonalizing the overlap matrix to extract the orthonormal basis
eigen, basis_change = jnp.linalg.eigh(overlaps)
orthogonal_basis = jnp.conjugate(basis_change.T) @ wave_fns
eigen,jnp.round(orthogonal_basis@(jnp.conjugate(orthogonal_basis.T)),2)

[[11018240.        +0.j    8420096.11-1105125.85j  8420096.11-1105125.85j]
 [ 8420096.11+1105125.85j  8676551.92      +0.j    5972271.73      +0.j  ]
 [ 8420096.11+1105125.85j  5972271.73      -0.j    8676551.92      +0.j  ]]


(Array([  687177.32249848,  2704280.19546607, 24979886.32380833], dtype=float64),
 Array([[  687177.32+0.j,       -0.  +0.j,        0.  +0.j],
        [      -0.  -0.j,  2704280.2 +0.j,       -0.  -0.j],
        [       0.  -0.j,       -0.  +0.j, 24979886.32+0.j]],      dtype=complex128))

In [67]:
# normalized wave_functions, divide each component of wave function by sqrt norm
# i.e. the sqrt eigenvalues eigen above
orthogonal_basis = (jnp.conjugate(basis_change.T) @ wave_fns)/jnp.sqrt(eigen[:,None])

jnp.round(orthogonal_basis@(jnp.conjugate(orthogonal_basis.T)),2)

Array([[ 1.+0.j, -0.-0.j,  0.-0.j],
       [-0.+0.j,  1.+0.j, -0.-0.j],
       [ 0.+0.j, -0.+0.j,  1.+0.j]], dtype=complex128)

In [63]:
jnp.round(jnp.conjugate(basis_change.T)/(eigen[:,None]),2)

Array([[-11.63+0.j  ,   6.18-0.81j,   6.18-0.81j],
       [  0.  -0.j  ,   2.27-0.12j,  -2.27+0.12j],
       [ -0.23+0.j  ,  -0.21+0.03j,  -0.21+0.03j]], dtype=complex128)

In [19]:
import jax.numpy as jnp

def wilson_loop(G):
    """
    Gauge-invariant loop
    1 -> 2 -> 4 -> 3 -> 1
    (Python indexing: 0 -> 1 -> 3 -> 2 -> 0)
    """
    W = G[0,1] * G[1,3] * G[3,2] * G[2,0]

    return {
        "W": W,
        "abs": jnp.abs(W),
        "phase": jnp.angle(W)
    }

In [20]:
wilson_loop(overlaps)

{'W': Array(0.39004906+1.04517873e-16j, dtype=complex128),
 'abs': Array(0.39004906, dtype=float64),
 'phase': Array(2.67960834e-16, dtype=float64)}

In [22]:
def triangle(G, i, j, k):
    B = G[i,j] * G[j,k] * G[k,i]
    return {
        "B": B,
        "abs": jnp.abs(B),
        "phase": jnp.angle(B)
    }

for tri in [(0,1,2),
            (0,1,3),
            (0,2,3),
            (1,2,3)]:
    print(tri, triangle(overlaps,*tri))

(0, 1, 2) {'B': Array(0.51925971+2.29229173e-17j, dtype=complex128), 'abs': Array(0.51925971, dtype=float64), 'phase': Array(4.41453805e-17, dtype=float64)}
(0, 1, 3) {'B': Array(0.42099779+0.07757745j, dtype=complex128), 'abs': Array(0.42808574, dtype=float64), 'phase': Array(0.18222626, dtype=float64)}
(0, 2, 3) {'B': Array(0.42099779+0.07757745j, dtype=complex128), 'abs': Array(0.42808574, dtype=float64), 'phase': Array(0.18222626, dtype=float64)}
(1, 2, 3) {'B': Array(0.35589296-7.29791374e-17j, dtype=complex128), 'abs': Array(0.35589296, dtype=float64), 'phase': Array(-2.05059235e-16, dtype=float64)}


In [24]:
jnp.pi/16 # compare with the 0.18 above

0.19634954084936207

In [68]:
import jax.numpy as jnp

# ---------------------------
# Compute transformation A
# ---------------------------
def compute_change_of_basis(G, G_target, eps=1e-12):
    """
    Returns A such that:
        A G A^† ≈ G_target
    """

    # Cholesky decompositions (G must be Hermitian PSD)
    L = jnp.linalg.cholesky(G + eps * jnp.eye(G.shape[0]))
    L_t = jnp.linalg.cholesky(G_target + eps * jnp.eye(G_target.shape[0]))

    # Inverse of L
    L_inv = jnp.linalg.inv(L)

    # Transformation
    A = L_t @ L_inv

    return A


# ---------------------------
# Apply transformation to vectors
# ---------------------------
def transform_vectors(V, A):
    """
    V: (N, d) matrix where rows are original vectors v_i
    A: (N, N) mixing matrix

    returns w_a = sum_i A[a,i] v_i
    """
    return A @ V


# ---------------------------
# Verify overlap
# ---------------------------
def compute_overlap(V):
    return V @ jnp.conjugate(V.T)


def test(G, G_target, V):
    A = compute_change_of_basis(G, G_target)
    V_new = transform_vectors(V, A)
    G_new = compute_overlap(V_new)
    return A, G_new

In [71]:
#overlap of un-normalized wave functions
overlaps = wave_fns@(jnp.conjugate(wave_fns.T))
# diagonal elements of over-laps are the norms of wave functions
norms = jnp.sqrt(jnp.diag(overlaps))
# normalized wave_functions, divide each component of wave function by sqrt norm
wave_fns = wave_fns/norms[:,None]
#overlap of normalized wave functions
overlaps = wave_fns@(jnp.conjugate(wave_fns.T))
G = jnp.round(overlaps,3)

In [79]:
sqrt2inv = float(1/jnp.sqrt(2))
Gtarget = jnp.array([[0.5,0.5,sqrt2inv],
                     [0.5,0.5,-sqrt2inv],
                     [sqrt2inv,-sqrt2inv,0]])
    
ChangeOfBasis = compute_change_of_basis(G, Gtarget)

In [80]:
ChangeOfBasis

Array([[nan+nanj, nan+nanj, nan+nanj],
       [nan+nanj, nan+nanj, nan+nanj],
       [nan+nanj, nan+nanj, nan+nanj]], dtype=complex128)

In [82]:
jnp.linalg.cholesky(G + 1e-12 * jnp.eye(G.shape[0]))

Array([[ 1.        +0.j   ,  0.        +0.j   ,  0.        +0.j   ],
       [ 0.861     +0.113j,  0.49589313+0.j   ,  0.        +0.j   ],
       [ 0.861     +0.113j, -0.13327468+0.j   ,  0.47764826+0.j   ]],      dtype=complex128)

In [84]:
jnp.linalg.cholesky(Gtarget + 1e-12 * jnp.eye(Gtarget.shape[0]))

Array([[nan,  0.,  0.],
       [nan, nan,  0.],
       [nan, nan, nan]], dtype=float64)

In [85]:
jnp.linalg.eigh(Gtarget)

EighResult(eigenvalues=Array([-1.,  1.,  1.], dtype=float64), eigenvectors=Array([[-0.5       , -0.8660254 ,  0.        ],
       [ 0.5       , -0.28867513, -0.81649658],
       [ 0.70710678, -0.40824829,  0.57735027]], dtype=float64))

In [86]:
import jax.numpy as jnp


# ---------------------------
# Spectral square root
# ---------------------------
def matrix_sqrt(G, eps=1e-12):
    evals, evecs = jnp.linalg.eigh(G)

    evals = jnp.clip(evals, a_min=eps)
    sqrt_evals = jnp.sqrt(evals)

    return (evecs * sqrt_evals) @ evecs.conj().T


# ---------------------------
# Spectral inverse square root
# ---------------------------
def matrix_inv_sqrt(G, eps=1e-12):
    evals, evecs = jnp.linalg.eigh(G)

    evals = jnp.clip(evals, a_min=eps)
    inv_sqrt_evals = 1.0 / jnp.sqrt(evals)

    return (evecs * inv_sqrt_evals) @ evecs.conj().T


# ---------------------------
# Build transformation A
# ---------------------------
def compute_A_spectral(G, G_target, eps=1e-12):
    G_inv_sqrt = matrix_inv_sqrt(G, eps)
    Gt_sqrt = matrix_sqrt(G_target, eps)

    A = Gt_sqrt @ G_inv_sqrt
    return A


# ---------------------------
# Apply to vectors
# ---------------------------
def transform_vectors(V, A):
    return A @ V


# ---------------------------
# Verify overlap
# ---------------------------
def overlap(V):
    return V @ jnp.conjugate(V.T)

In [88]:
jnp.round(compute_A_spectral(G, Gtarget),3)

Array([[ 1.335-0.086j, -0.364+0.106j, -0.178+0.106j],
       [ 0.234-0.056j,  1.072+0.035j, -0.904+0.035j],
       [ 0.778-0.021j, -1.015+0.05j ,  0.513+0.05j ]], dtype=complex128)